# 0. WATERec options and experiment findings

Start with `README.md`, `docs/capabilities.md` and `docs/experiments.md`. R and RS are equal-size pretrained variants (26.12M parameters), not a Nano/Tiny size ladder. Price SPACE adds 768 parameters. PARSeq Fast Tiny is maintained in its separate repository.

Our qualified plate profile is RS/512; the detailed-price profile is R/2048. Their checkpoints, training pools and timing scopes differ. The original 17.9M custom terminal reached 159/206 raw and 186/206 numeric; the later restart2 terminal reaches 161/206 raw and 187/206 numeric. Both retain 132/144 AI-reviewed numeric, one case below Florence. A separate experimental compact workflow is available in notebook 9; the reference weights remain unbundled and the standard R/RS options are unchanged. Dummy examples establish workflow behavior only.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
reference = json.loads((ROOT / "results/reference_metrics.json").read_text())
print(json.dumps(reference["model_options"], indent=2))


Choose `WATEREC_CONFIG=configs/dummy_price.yaml` for fictional mixed-size prices. Real-data starting profiles are `waterec_alpr.yaml`, `waterec_r_alpr.yaml`, `waterec_alpr_1024.yaml`, `waterec_price.yaml`, and `waterec_price_512.yaml`. Changing budget requires separate quality checks and matching exports. Default native labels are at most 24 characters with 25 decoder steps including EOS.

## Historical compact research and the experimental workflow

See [the compact research update](../docs/compact_research.md) and [aggregate evidence](../results/compact_research.json). The image-only token selector uses 2048 input patches and 512 late encoder/decoder memory tokens, 17,913,953 parameters, and image/text-only training. For the original terminal, the earlier runtime measured 1.422 ms RGB-to-text and 708 MiB process GPU usage on RTX 5090, static batch 1. These are not edge-device measurements.

Raw 159/206, numeric 186/206 and AI-assisted numeric 132/144 are different metrics. The original strict numeric tolerance failed; a separate text-output qualification matched all 206 native strings. Reference weights remain unbundled; a separate experimental training/export/runtime workflow is now available in notebook 9. The standard R/RS configs below do not instantiate this model.

### Latest research terminal: restart2
The same 17.913953M-parameter architecture, a 2,048-patch input budget with 512 patches retained after six blocks, two autoregressive layers: **161/206 raw (78.16%), 187/206 full-price numeric (90.78%), 132/144 AI-reviewed numeric; 90.18% mean normalized Levenshtein similarity**. The earlier private restart2 runtime measured 1.362 ms RGB /1.620 ms encoded, 708 MiB GPU; current host 581.1 /549.7MiB. Five warmups, three 206-image passes, all outputs match the new native terminal.
This is not 90% raw exact or a lossless model change: 16 strings change, six raw gains and four losses. Original 0.05 intermediate numeric failures remain recorded; new through-EOS output fidelity is a separate policy, not routing/calibration equivalence. AI 144 is uncertified and Florence remains one case higher. See the linked research document for fixed 820-image training/restart provenance and hashes. **Standard R/RS model choices are unchanged; use the dedicated experimental workflow in notebook 9. Reference weights are not packaged.** No edge, batch or rotation claim.

Numeric equivalence compares the full price value, including its decimal fraction; raw exact also checks currency, punctuation and spacing.


### Separate experimental workflow is now available

Use [notebook 9](9_experimental_compact_workflow.ipynb) and [the workflow guide](../docs/compact_workflow.md) for multistage distillation, not the one-step Florence recipe. One-epoch defaults are software demos; teacher provenance is your responsibility, the charset is limited, and automatic resume is absent. The exact reference checkpoint's current standalone public runtime v3 measured **1.356083 ms RGB /1.620985 ms encoded, 708 MiB**, preserving all 206 native strings. These are not results for newly trained demo weights. Native resolution flexibility does not imply arbitrary TensorRT budgets, dynamic batching, edge or rotation qualification.
